# 📄 Document Image Classification with EfficientNetB0 — RVL-CDIP Mini

**Author:** [Mauro26-AI](https://github.com/Mauro26-AI)  
**Dataset:** [`dvgodoy/rvl_cdip_mini`](https://huggingface.co/datasets/dvgodoy/rvl_cdip_mini) (HuggingFace Hub)  
**Task:** Multi-class classification of scanned document images into 16 categories  
**Model:** EfficientNetB0 with two-phase Transfer Learning (frozen backbone → full fine-tuning)  
**Reference Paper:** Harley et al., *Evaluation of Deep Convolutional Nets for Document Image Classification and Retrieval*, ICDAR 2015

---

## Description

This project implements an end-to-end **Intelligent Document Processing (IDP)** pipeline for automatic classification of scanned office documents. Using a pre-trained EfficientNetB0 backbone fine-tuned on the RVL-CDIP Mini benchmark, the model learns to distinguish 16 document types — from invoices and advertisements to scientific publications and handwritten notes — directly from pixel-level image features, without any OCR preprocessing.

The training strategy follows a **two-phase transfer learning** protocol: the backbone is first frozen to train only the custom classification head, then fully unfrozen for low-learning-rate fine-tuning. This approach stabilises convergence on the small dataset (~3,200 training samples) while maximising the reuse of ImageNet visual representations.

---

## Notebook Structure

1. Setup & Installation  
2. Dataset Loading  
3. Exploratory Data Analysis (EDA)  
4. Preprocessing & Data Augmentation  
5. Model Architecture  
6. Training  
7. Evaluation & Metrics  
8. Error Analysis  
9. Model Saving & Inference  
10. Conclusions & Future Work

---
## 1. ⚙️ Setup & Installation

In [ ]:
# # Install required dependencies
# !pip install -r requirements.txt

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from datasets import load_dataset
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    top_k_accuracy_score,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# torch.cuda.manual_seed_all(SEED)
# torch.backends.cudnn.deterministic = True

# Device selection
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. 📥 Dataset Loading

In [ ]:
# Load the RVL-CDIP Mini dataset from HuggingFace Hub
# Splits: 3,200 train / 400 validation / 400 test
print("Loading dvgodoy/rvl_cdip_mini ...")
raw_dataset = load_dataset("dvgodoy/rvl_cdip_mini")
print(raw_dataset)
print("\nAvailable features:", raw_dataset["train"].features)

In [ ]:
# Build deterministic class → index mappings from the training split
train_ds = raw_dataset["train"]
val_ds   = raw_dataset["validation"]
test_ds  = raw_dataset["test"]

CLASS_NAMES = sorted(list(set(train_ds["category"])))
NUM_CLASSES = len(CLASS_NAMES)
class2idx   = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2class   = {i: c for c, i in class2idx.items()}

print(f"Number of classes : {NUM_CLASSES}")
print(f"Classes           : {CLASS_NAMES}")

---
## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution in the training set
train_labels = train_ds["category"]
label_counts  = Counter(train_labels)

fig, ax = plt.subplots(figsize=(14, 5))
classes_sorted = sorted(label_counts.keys())
counts_sorted  = [label_counts[c] for c in classes_sorted]

bars = ax.bar(classes_sorted, counts_sorted, color="steelblue", edgecolor="white")
ax.set_title("Class Distribution — Training Set", fontsize=14, fontweight="bold")
ax.set_xlabel("Document Category")
ax.set_ylabel("Number of Samples")
ax.set_xticklabels(classes_sorted, rotation=45, ha="right")
for bar, count in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

most_common  = max(label_counts, key=label_counts.get)
least_common = min(label_counts, key=label_counts.get)
balanced     = max(label_counts.values()) == min(label_counts.values())
print(f"Most common class  : {most_common} ({label_counts[most_common]})")
print(f"Least common class : {least_common} ({label_counts[least_common]})")
print(f"Balance            : {'BALANCED' if balanced else 'IMBALANCED'}")

In [ ]:
# One representative sample per class
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.flatten()

for i, class_name in enumerate(CLASS_NAMES):
    sample = next(s for s in train_ds if s["category"] == class_name)
    img = sample["image"].convert("RGB")
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(class_name, fontsize=11, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("One Sample per Class — RVL-CDIP Mini", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Image dimension analysis (original resolution before resizing)
widths  = train_ds["width"]
heights = train_ds["height"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths,  bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Width Distribution (px)")
axes[0].set_xlabel("Width")
axes[0].set_ylabel("Frequency")

axes[1].hist(heights, bins=30, color="coral", edgecolor="white")
axes[1].set_title("Height Distribution (px)")
axes[1].set_xlabel("Height")
axes[1].set_ylabel("Frequency")

axes[2].scatter(widths, heights, alpha=0.3, s=10, color="mediumseagreen")
axes[2].set_title("Width vs Height")
axes[2].set_xlabel("Width")
axes[2].set_ylabel("Height")

plt.suptitle("Image Resolution Statistics — Training Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f} px")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f} px")

In [ ]:
# Aspect ratio analysis
# Most document scans are portrait-oriented (width < height → aspect ratio < 1.0)
aspect_ratios = [w / h for w, h in zip(widths, heights)]
portrait_pct  = sum(1 for ar in aspect_ratios if ar < 1.0) / len(aspect_ratios) * 100

print(f"Aspect ratio — mean: {np.mean(aspect_ratios):.3f}, std: {np.std(aspect_ratios):.3f}")
print(f"Portrait orientation (aspect ratio < 1.0): {portrait_pct:.1f}% of training images")

---
## 4. 🛠️ Preprocessing & Data Augmentation

The preprocessing pipeline converts variable-resolution TIFF greyscale scans into
fixed-size normalised tensors compatible with the EfficientNetB0 pre-trained weights.

**Augmentation design rationale:** Transforms are chosen to simulate realistic scanner
variability without destroying the structural layout of documents — the key visual signal
for classification.

In [ ]:
# Get weights_EfficientNet for normalization
weights_EfficientNet = models.EfficientNet_B0_Weights.DEFAULT
transforms_norm = weights_EfficientNet.transforms()
print(transforms_norm)

In [ ]:
# Global hyperparameters
IMG_SIZE    = 224   # EfficientNetB0 expected input resolution
BATCH_SIZE  = 32
NUM_EPOCHS  = 20    # Phase 1 (frozen backbone)
LR          = 1e-3  # Phase 1 learning rate
LR_FINETUNE = 1e-4  # Phase 2 learning rate (full fine-tuning)
PATIENCE    = 5     # Early stopping patience (epochs to wait without val improvement)

In [ ]:
# ImageNet normalisation statistics
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Training pipeline: augmentation + normalization
train_transform = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),    
    T.RandomCrop(IMG_SIZE),                      
    T.RandomRotation(degrees=5),                 
    T.RandomHorizontalFlip(p=0.1),                
    T.ColorJitter(brightness=0.3, contrast=0.3),  
    T.Grayscale(num_output_channels=3),           # greyscale scan → 3-channel RGB-style tensor
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

# Validation / test pipeline: deterministic centre resize only
val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

In [ ]:
class RVLDataset(Dataset):
    """
    PyTorch Dataset wrapper around a HuggingFace DatasetDict split.

    Converts PIL images (possibly TIFF greyscale) to RGB, applies
    the provided transform pipeline, and returns (tensor, label_index) pairs.
    """

    def __init__(self, hf_dataset, transform=None):
        self.data      = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image  = sample["image"].convert("RGB")   # TIFF L-mode → 3-channel RGB
        label  = class2idx[sample["category"]]
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = RVLDataset(train_ds, transform=train_transform)
val_dataset   = RVLDataset(val_ds,   transform=val_transform)
test_dataset  = RVLDataset(test_ds,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
# Visualise a batch of training samples after augmentation
# PyTorch tensors are [C, H, W]; Matplotlib expects [H, W, C] → permute(1, 2, 0)
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array(STD) + np.array(MEAN) 
    img = np.clip(img, 0, 1)
    ax.imshow(img[:, :, 0], cmap="gray")
    ax.set_title(idx2class[labels[i].item()], fontsize=7)
    ax.axis("off")

plt.suptitle("Training Batch Samples (post-augmentation)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. 🏗️ Model Architecture — EfficientNetB0 + Custom Head

**Why EfficientNetB0?**  
EfficientNet models achieve state-of-the-art accuracy with significantly fewer parameters
than equivalently-performing ResNet or VGG architectures, thanks to compound scaling of
depth, width, and resolution. EfficientNetB0 (~4.67M parameters) provides an excellent
accuracy/efficiency trade-off for small datasets (~3,200 training samples), where larger
backbones would overfit.

Pre-trained on ImageNet, its convolutional features (edges, textures, shapes) transfer
effectively to document images, which share similar low-level visual primitives.

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    Build an EfficientNetB0 model with a custom classification head.
    Returns:
        Configured PyTorch model.
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model


model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters              : {total_params:>10,}")
print(f"Trainable parameters (Phase 1): {trainable_params:>10,}  ({trainable_params / total_params * 100:.1f}%)")

---
## 6. 🚀 Training

### Two-Phase Transfer Learning Strategy

#### Phase 1 — Frozen Backbone
The EfficientNetB0 backbone already encodes rich visual representations from ImageNet
(edges, textures, shapes). In Phase 1, those weights are frozen and only the randomly
initialised classification head is trained. This is fast, stable, and within a few
epochs the head learns a solid mapping from features to document categories.

#### Phase 2 — Full Fine-tuning
Once the head is trained and stable, all parameters are unfrozen and the entire model
is retrained with a very low learning rate (`1e-4`). This allows the backbone to
specialise its representations for document images while the small learning rate
prevents catastrophic forgetting of the ImageNet features.

In [ ]:
# Loss function, optimiser, and learning rate scheduler for Phase 1
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """
    Run one full training epoch.

    Returns:
        (average_loss, accuracy) over the entire training loader.
    """
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, list, list, np.ndarray]:
    """
    Evaluate the model on a data loader.

    Returns:
        (average_loss, accuracy, predicted_labels, true_labels, probability_matrix)
    """
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = probs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    prob_matrix = np.vstack(all_probs)
    return running_loss / total, correct / total, all_preds, all_labels, prob_matrix

In [ ]:
def run_training(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: optim.Optimizer,
    scheduler,
    criterion: nn.Module,
    device: torch.device,
    num_epochs: int,
    patience: int,
    phase_name: str = "Phase 1",
) -> dict:
    """
    Full training loop with early stopping and best-weight restoration.

    Tracks training/validation loss and accuracy per epoch.
    Restores the weights from the best validation accuracy epoch.

    Returns:
        history dict with keys: train_loss, val_loss, train_acc, val_acc, best_epoch.
    """
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None
    best_epoch   = 0
    no_improve   = 0

    print(f"\n{'=' * 65}")
    print(f"  {phase_name}")
    print(f"{'=' * 65}")

    for epoch in range(1, num_epochs + 1):
        t_loss, t_acc  = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_acc, _, _, _  = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)

        flag = ""
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_epoch = epoch
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            flag = "  ★ best"
        else:
            no_improve += 1

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:>3}/{num_epochs} | "
            f"train loss: {t_loss:.4f}  acc: {t_acc:.4f} | "
            f"val loss: {v_loss:.4f}  acc: {v_acc:.4f} | "
            f"lr: {lr_now:.2e}{flag}"
        )

        if no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch} epochs ({patience} epochs without improvement).")
            break

    # Restore best checkpoint
    model.load_state_dict(best_weights)
    print(f"\nBest validation accuracy: {best_val_acc:.4f}  (epoch {best_epoch})")
    history["best_epoch"]    = best_epoch
    history["best_val_acc"]  = best_val_acc
    return history

In [ ]:
# Phase 1: train the classification head only (backbone frozen)
history_phase1 = run_training(
    model, train_loader, val_loader,
    optimizer, scheduler, criterion,
    DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    phase_name="Phase 1 — Frozen Backbone, Head Training",
)

In [ ]:
# Phase 2: full fine-tuning (all parameters unfrozen)
for param in model.parameters():
    param.requires_grad = True

optimizer_ft = optim.Adam(model.parameters(), lr=LR_FINETUNE)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=10)

total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters (Phase 2): {total_trainable:,}  (100%)")

history_phase2 = run_training(
    model, train_loader, val_loader,
    optimizer_ft, scheduler_ft, criterion,
    DEVICE,
    num_epochs=10,
    patience=5,
    phase_name="Phase 2 — Full Fine-tuning",
)

In [ ]:
def plot_history(h1: dict, h2: dict) -> None:
    """
    Plot combined training curves for both phases.
    Phase 1 is shown in blue, Phase 2 in red; a vertical dashed line marks
    the transition between phases.
    """
    n1 = len(h1["train_loss"])
    n2 = len(h2["train_loss"])
    epochs1 = range(1, n1 + 1)
    epochs2 = range(n1 + 1, n1 + n2 + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].plot(epochs1, h1["train_loss"], "b-",  label="Train (Phase 1)")
    axes[0].plot(epochs1, h1["val_loss"],   "b--", label="Val   (Phase 1)")
    axes[0].plot(epochs2, h2["train_loss"], "r-",  label="Train (Phase 2)")
    axes[0].plot(epochs2, h2["val_loss"],   "r--", label="Val   (Phase 2)")
    axes[0].axvline(x=n1, color="gray", linestyle=":", label="Fine-tuning start")
    axes[0].set_title("Learning Curves — Cross-Entropy Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy curves
    axes[1].plot(epochs1, h1["train_acc"], "b-",  label="Train (Phase 1)")
    axes[1].plot(epochs1, h1["val_acc"],   "b--", label="Val   (Phase 1)")
    axes[1].plot(epochs2, h2["train_acc"], "r-",  label="Train (Phase 2)")
    axes[1].plot(epochs2, h2["val_acc"],   "r--", label="Val   (Phase 2)")
    axes[1].axvline(x=n1, color="gray", linestyle=":", label="Fine-tuning start")
    axes[1].set_title("Learning Curves — Accuracy", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f"Training History | Phase 1 best epoch: {h1['best_epoch']} | Phase 2 best epoch: {h2['best_epoch']}",
                 fontsize=11)
    plt.tight_layout()
    plt.show()


plot_history(history_phase1, history_phase2)

---
## 7. 📊 Evaluation & Metrics

In [ ]:
# Evaluate on the held-out test set
test_loss, test_acc, y_pred, y_true, y_probs = evaluate(model, test_loader, criterion, DEVICE)

macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
top3_acc = top_k_accuracy_score(y_true, y_probs, k=3)
top5_acc = top_k_accuracy_score(y_true, y_probs, k=5)

print(f"\n{'=' * 55}")
print(f"  TEST SET RESULTS")
print(f"{'=' * 55}")
print(f"  Test Loss          : {test_loss:.4f}")
print(f"  Top-1 Accuracy     : {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"  Top-3 Accuracy     : {top3_acc:.4f}  ({top3_acc * 100:.2f}%)")
print(f"  Top-5 Accuracy     : {top5_acc:.4f}  ({top5_acc * 100:.2f}%)")
print(f"  Macro F1-score     : {macro_f1:.4f}")
print(f"  Weighted F1-score  : {weighted_f1:.4f}")
print(f"{'=' * 55}")

In [ ]:
# Per-class precision, recall, and F1-score
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Confusion matrix: raw counts and row-normalised (recall per class)
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

for ax, data, title, fmt in [
    (axes[0], cm,      "Confusion Matrix (counts)",      "d"),
    (axes[1], cm_norm, "Confusion Matrix (normalised)",  ".2f"),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5, cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Ground Truth", fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy (diagonal of the normalised confusion matrix)
# Colour-coded: red < 60% | orange 60–80% | green ≥ 80%
per_class_acc = cm_norm.diagonal()
sorted_idx    = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = [
    "#d9534f" if per_class_acc[i] < 0.60 else
    "#f0ad4e" if per_class_acc[i] < 0.80 else
    "#5cb85c"
    for i in sorted_idx
]
bars = ax.barh(
    [CLASS_NAMES[i] for i in sorted_idx],
    [per_class_acc[i] for i in sorted_idx],
    color=bar_colors,
)
ax.axvline(x=0.60, color="red",    linestyle="--", alpha=0.6, label="60% threshold")
ax.axvline(x=0.80, color="orange", linestyle="--", alpha=0.6, label="80% threshold")
ax.set_title("Per-Class Accuracy — Test Set", fontsize=13, fontweight="bold")
ax.set_xlabel("Accuracy")
ax.set_xlim(0, 1.1)
ax.legend()
for bar, val in zip(bars, [per_class_acc[i] for i in sorted_idx]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 8. 🔬 Error Analysis

In [ ]:
# Visualise misclassified test samples
y_pred_arr = np.array(y_pred)
y_true_arr = np.array(y_true)
wrong_idx  = np.where(y_pred_arr != y_true_arr)[0]

error_rate = len(wrong_idx) / len(y_true_arr) * 100
print(f"Misclassified samples: {len(wrong_idx)} / {len(y_true_arr)}  ({error_rate:.1f}% error rate)")

n_show = min(8, len(wrong_idx))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(n_show):
    sample_idx = wrong_idx[i]
    sample     = test_ds[int(sample_idx)]
    img        = sample["image"].convert("RGB")
    true_label = idx2class[y_true_arr[sample_idx]]
    pred_label = idx2class[y_pred_arr[sample_idx]]

    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(
        f"Ground truth: {true_label}\nPredicted:    {pred_label}",
        fontsize=9,
        color="crimson",
    )
    axes[i].axis("off")

plt.suptitle("Misclassified Test Samples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Top-10 most confused class pairs (sorted by error count)
confusion_pairs = [
    (CLASS_NAMES[i], CLASS_NAMES[j], cm[i, j])
    for i in range(NUM_CLASSES)
    for j in range(NUM_CLASSES)
    if i != j and cm[i, j] > 0
]
confusion_pairs.sort(key=lambda x: -x[2])

print(f"{'Ground Truth':<28} {'Predicted As':<28} {'Errors':>6}")
print("-" * 65)
for true_c, pred_c, count in confusion_pairs[:10]:
    print(f"{true_c:<28} {pred_c:<28} {count:>6}")

---
## 9. 💾 Model Saving & Inference

In [ ]:
# Save model weights together with metadata required for inference
checkpoint = {
    "model_state_dict" : model.state_dict(),
    "class_names"      : CLASS_NAMES,
    "class2idx"        : class2idx,
    "img_size"         : IMG_SIZE,
    "test_accuracy"    : test_acc,
    "top3_accuracy"    : top3_acc,
    "macro_f1"         : macro_f1,
    "training_config"  : {
        "phase1_best_epoch" : history_phase1["best_epoch"],
        "phase1_best_val"   : history_phase1["best_val_acc"],
        "phase2_best_epoch" : history_phase2["best_epoch"],
        "phase2_best_val"   : history_phase2["best_val_acc"],
        "lr_phase1"         : LR,
        "lr_phase2"         : LR_FINETUNE,
    },
}

checkpoint_path = "rvl_cdip_efficientnet_b0_.pt"
torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")

In [ ]:
def predict_image(
    image_input,
    model: nn.Module,
    transform,
    device: torch.device,
    top_k: int = 5,
) -> tuple[str, np.ndarray]:
    """
    Classify a single document image and return the predicted label with probabilities.

    Args:
        image_input:  File path (str) or PIL Image object.
        model:        Trained classification model.
        transform:    Inference transform pipeline (val_transform).
        device:       Torch device.
        top_k:        Number of top predictions to display.

    Returns:
        (predicted_class_name, probability_array)
    """
    if isinstance(image_input, str):
        img = Image.open(image_input).convert("RGB")
    else:
        img = image_input.convert("RGB")

    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()

    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
        pred   = probs.argmax().item()

    topk_probs, topk_idx = probs.topk(top_k)

    print(f"\nPrediction : {idx2class[pred]}  ({probs[pred] * 100:.2f}%)")
    print(f"\nTop-{top_k} predictions:")
    for p, i in zip(topk_probs, topk_idx):
        print(f"  {idx2class[i.item()]:30s}  {p.item() * 100:.2f}%")

    return idx2class[pred], probs.cpu().numpy()


# Inference example on the first test sample
sample_img  = test_ds[0]["image"]
true_label  = test_ds[0]["category"]
print(f"Ground truth: {true_label}")
predict_image(sample_img, model, val_transform, DEVICE);

---
## 10. 📝 Conclusions & Future Work

This project is a **research prototype**, trained on a
mini benchmark (~200 samples per class), it establishes the foundational
methodology for document classification pipelines — but bridging the gap to deployment
would require the full RVL-CDIP corpus, GPU-accelerated inference, and integration with
a real document ingestion stack.

The core technique demonstrated here — automatic document-type routing from visual layout
alone — is the first stage of any **Intelligent Document Processing (IDP)** system, with
direct applications in invoice automation, enterprise archive indexing, and legal-tech
document triage. The logical next step is a multi-modal extension via **LayoutLM**,
combining visual features with the OCR tokens already present in this dataset to address
the confusion pairs observed in evaluation.

### Strengths

- **Two-phase transfer learning:** Freezing the backbone before full fine-tuning ensures
  stable convergence on a small dataset (~3,200 training images) while maximising reuse
  of ImageNet representations.
- **Domain-aware augmentation:** Transforms are calibrated to simulate realistic scanner
  variability (slight rotation, brightness/contrast jitter) without distorting the
  structural layout that carries discriminative information for document classification.
- **EfficientNetB0:** Optimal accuracy-to-parameter trade-off for small-scale experiments;
  only 14.2% of parameters are trained in Phase 1, keeping Phase 1 fast and stable.

---

### Limitations

- **Small training set:** ~200 samples per class is an objectively challenging regime for
  deep CNNs. The full RVL-CDIP corpus (400,000 images) would unlock significantly better
  performance without architectural changes.
- **Resolution loss:** Original images are high-resolution TIFF scans downsampled to
  224×224, discarding fine textual details that are critical for visually similar classes
  such as `handwritten` vs `letter`.
- **Visually similar class pairs:** Pairs such as `memo`/`letter`,
  `scientific report`/`scientific publication`, and `form`/`questionnaire` share nearly
  identical page layouts. Both the model and a human annotator are expected to confuse
  these pairs without reading the actual text content.

---

### Future Work

| Direction | Description | Expected Impact |
|---|---|---|
| **Full dataset** | Use `aharley/rvl_cdip` (400K images) | Large accuracy gain |
| **Vision Transformers** | ViT / DiT (`microsoft/dit-base-finetuned-rvlcdip`) | SOTA on this benchmark |
| **Multi-modal fusion** | Combine visual features + OCR tokens (LayoutLM architecture) | Better on text-heavy classes |
| **Test-Time Augmentation** | Average predictions over multiple crops/flips | +1–2% robustness |
| **Grad-CAM** | Visualise which document regions activate the classifier | Interpretability |
| **Weighted loss** | Handle class imbalance in the full dataset | Improved minority-class recall |